# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

In [ ]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'daily'
DATA_PATH = DATA_DIR / '*.csv'

csv_files = list(DATA_DIR.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(
        f'Dataset tidak ditemukan di: {DATA_DIR}. '
        'Pastikan data telah diekstrak ke data/raw/daily/.'
    )

total_size_mb = sum(f.stat().st_size for f in csv_files) / 1_000_000
print(f'Folder Dataset: {DATA_DIR.name}')
print(f'Jumlah file: {len(csv_files):,} file CSV emiten')
print(f'Total ukuran file mentah: {total_size_mb:.1f} MB')

## Profil Polars

`scan_csv` dengan wildcard hanya membangun rencana query; data baru dibaca secara efisien dan paralel ketika `collect()` dipanggil.

In [ ]:
lazy_data = pl.scan_csv(
    str(DATA_PATH),
    schema_overrides={
        'open': pl.Float64,
        'low': pl.Float64,
        'high': pl.Float64,
        'close': pl.Float64,
        'volume': pl.Float64,
    },
    ignore_errors=True,
    include_file_paths='file_path',
).with_columns(
    pl.col('file_path')
    .str.split('/')
    .list.last()
    .str.split('\\')
    .list.last()
    .str.replace(r'\.csv$', '')
    .alias('ticker')
).drop('file_path')

print(lazy_data.collect_schema())

ringkasan = lazy_data.select([
    pl.len().alias('jumlah_baris'),
    pl.all().null_count().name.suffix('_null'),
]).collect()
ringkasan

In [ ]:
contoh_data = lazy_data.head(10).collect()
contoh_data

## Profil DuckDB

DuckDB membaca file CSV langsung dari disk secara paralel dan menghasilkan ringkasan statistik setiap kolom.

In [ ]:
connection = duckdb.connect()
duckdb_path = str(DATA_PATH).replace('\\', '/')
profil_duckdb = connection.execute(
    'SUMMARIZE SELECT * FROM read_csv_auto(?)', [duckdb_path]
).pl()
profil_duckdb

## Catatan Milestone 1

Tulis temuan awal di bawah ini: ukuran data, jumlah baris, tipe kolom, kolom dengan null terbanyak, serta anomali yang perlu ditangani pada Milestone 2.